In [1]:
%%capture
!pip install virtualizarr obstore icechunk

In [2]:
import icechunk
import obstore
import xarray as xr

from virtualizarr import open_virtual_dataset, VirtualiZarrDataTreeAccessor
from virtualizarr.parsers import HDFParser
from virtualizarr.registry import ObjectStoreRegistry

import ismip6_helper

In [3]:
bucket = "gs://ismip6"
store = obstore.store.from_url(bucket, skip_signature=True)
registry = ObjectStoreRegistry({bucket: store})

In [4]:
ismip6_df = ismip6_helper.get_file_index()

Loading index from cache: .cache/ismip6_index.parquet


In [5]:
# For the purposes of this demo, filter down the number of files we have to load
ismip6_df_filtered = ismip6_df.query('experiment in ["ctrl_proj_std", "exp05", "ctrl_proj"] and variable in ["lithk", "base", "sftgrf"] and institution in ["JPL1", "AWI", "DOE"]')

,variable,ice_sheet,institution,model_name,experiment,url,size_bytes
1,base,AIS,AWI,PISM1,ctrl_proj_std,gs://ismip6/Projection-AIS/AWI/PISM1/ctrl_proj...,61015910
12,lithk,AIS,AWI,PISM1,ctrl_proj_std,gs://ismip6/Projection-AIS/AWI/PISM1/ctrl_proj...,59188306
16,sftgrf,AIS,AWI,PISM1,ctrl_proj_std,gs://ismip6/Projection-AIS/AWI/PISM1/ctrl_proj...,5619821
141,base,AIS,AWI,PISM1,exp05,gs://ismip6/Projection-AIS/AWI/PISM1/exp05/bas...,60864380
152,lithk,AIS,AWI,PISM1,exp05,gs://ismip6/Projection-AIS/AWI/PISM1/exp05/lit...,59022963
156,sftgrf,AIS,AWI,PISM1,exp05,gs://ismip6/Projection-AIS/AWI/PISM1/exp05/sft...,5565404
1018,base,AIS,DOE,MALI,ctrl_proj_std,gs://ismip6/Projection-AIS/DOE/MALI/ctrl_proj_...,231665144
1028,lithk,AIS,DOE,MALI,ctrl_proj_std,gs://ismip6/Projection-AIS/DOE/MALI/ctrl_proj_...,231665216
1032,sftgrf,AIS,DOE,MALI,ctrl_proj_std,gs://ismip6/Projection-AIS/DOE/MALI/ctrl_proj_...,231665220
1043,base,AIS,DOE,MALI,exp05,gs://ismip6/Projection-AIS/DOE/MALI/exp05/base...,199234312


In [6]:
# Build a DataTree of the outputs
datasets = {}
parser = HDFParser()
for _, row in ismip6_df_filtered.iterrows():
    try:
        p = f'{row["institution"]}_{row["model_name"]}/{row["experiment"]}/{row["variable"]}' # DataTree path
        
        vds = open_virtual_dataset(
          url=row["url"],
          parser=parser,
          registry=registry,
          loadable_variables=[],
        )
        vds_fix_time = ismip6_helper.fix_time_encoding(vds)
        vds_fix_coords = ismip6_helper.correct_grid_coordinates(vds_fix_time, row["variable"])

        datasets[p] = vds_fix_coords
    except Exception as e:
        print(f"Failed to load {p}: {e}")

/home/jupyter/conda_env/python313/lib/python3.13/site-packages/zarr/codecs/numcodecs/_codecs.py:141: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)
/home/jupyter/conda_env/python313/lib/python3.13/site-packages/zarr/codecs/numcodecs/_codecs.py:141: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)
/home/jupyter/conda_env/python313/lib/python3.13/site-packages/zarr/codecs/numcodecs/_codecs.py:141: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)
/home/jupyter/conda_env/python313/lib/python3.13/site-packages/zarr/codecs/numcodecs/_codecs.py:141: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not

⚠️  Grid correction: Dataset missing x/y coordinates for 'base'
   Detected dimensions: y=761, x=761
   Estimated resolution: dx=8.0 km, dy=8.0 km
   Creating coordinates: x=[-3040.0, 3040.0] km, y=[-3040.0, 3040.0] km
   ✓ Grid correction complete

⚠️  Grid correction: Dataset missing x/y coordinates for 'lithk'
   Detected dimensions: y=761, x=761
   Estimated resolution: dx=8.0 km, dy=8.0 km
   Creating coordinates: x=[-3040.0, 3040.0] km, y=[-3040.0, 3040.0] km
   ✓ Grid correction complete

⚠️  Grid correction: Dataset missing x/y coordinates for 'sftgrf'
   Detected dimensions: y=761, x=761
   Estimated resolution: dx=8.0 km, dy=8.0 km
   Creating coordinates: x=[-3040.0, 3040.0] km, y=[-3040.0, 3040.0] km
   ✓ Grid correction complete

⚠️  Grid correction: Dataset missing x/y coordinates for 'base'
   Detected dimensions: y=761, x=761
   Estimated resolution: dx=8.0 km, dy=8.0 km
   Creating coordinates: x=[-3040.0, 3040.0] km, y=[-3040.0, 3040.0] km
   ✓ Grid correction complet

In [7]:
ismip6_dt = xr.DataTree.from_dict(datasets)

In [8]:
vzdt = VirtualiZarrDataTreeAccessor(ismip6_dt)

In [12]:
# Configure the virtual chunk container for GCS
config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        "gs://ismip6/",
        store=icechunk.gcs_store()
    )
)

# When opening the repo, use None for anonymous/public access
credentials = icechunk.containers_credentials({
    "gs://ismip6/": None  # None uses anonymous credentials if allowed
})

icechunk_storage = icechunk.in_memory_storage()
repo = icechunk.Repository.create(icechunk_storage, config, authorize_virtual_chunk_access=credentials)
session = repo.writable_session("main")
vzdt.to_icechunk(session.store)
session.commit("Create virtual store")

'FZXZR00A8NBPGMP85V8G'

In [13]:
repo = icechunk.Repository.open(
    storage=icechunk_storage,
    config=config,
    authorize_virtual_chunk_access=credentials
)

Now, let's see if we can read from it

In [14]:
vzdt = xr.open_datatree(session.store, engine='zarr')
vzdt

/home/jupyter/conda_env/python313/lib/python3.13/site-packages/zarr/codecs/numcodecs/_codecs.py:141: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)


<xarray.DataTree>
Group: /
├── Group: /AWI_PISM1
│   ├── Group: /AWI_PISM1/ctrl_proj_std
│   │   ├── Group: /AWI_PISM1/ctrl_proj_std/base
│   │   │       Dimensions:    (time: 86, bnds: 2, y: 761, x: 761)
│   │   │       Coordinates:
│   │   │         * time       (time) object 688B 2016-01-01 00:00:00 ... 2101-01-01 00:00:00
│   │   │         * bnds       (bnds) float32 8B 0.0 0.0
│   │   │         * y          (y) float64 6kB -3.04e+06 -3.032e+06 ... 3.032e+06 3.04e+06
│   │   │         * x          (x) float64 6kB -3.04e+06 -3.032e+06 ... 3.032e+06 3.04e+06
│   │   │       Data variables:
│   │   │           time_bnds  (time, bnds) float32 688B ...
│   │   │           base       (time, y, x) float32 199MB ...
│   │   │       Attributes:
│   │   │           Conventions:  CF-1.6
│   │   │           title:        ISMIP6 Projections Greenland model output
│   │   │           institution:  Alfred Wegener Institute for Polar and Marine Research, DE,...
│   │   │           source:       PISM
│   │   │           references:   https://doi.org/10.5194/tc-14-3033-2020
│   │   │           contact:      Name = Thomas Kleiner, Johannes Sutter, Angelika Humbert, E...
│   │   │           comment:      AWI, PISM1
│   │   ├── Group: /AWI_PISM1/ctrl_proj_std/lithk
│   │   │       Dimensions:    (time: 86, y: 761, x: 761, bnds: 2)
│   │   │       Coordinates:
│   │   │         * time       (time) object 688B 2016-01-01 00:00:00 ... 2101-01-01 00:00:00
│   │   │         * y          (y) float64 6kB -3.04e+06 -3.032e+06 ... 3.032e+06 3.04e+06
│   │   │         * x          (x) float64 6kB -3.04e+06 -3.032e+06 ... 3.032e+06 3.04e+06
│   │   │         * bnds       (bnds) float32 8B 0.0 0.0
│   │   │       Data variables:
│   │   │           lithk      (time, y, x) float32 199MB ...
│   │   │           time_bnds  (time, bnds) float32 688B ...
│   │   │       Attributes:
│   │   │           Conventions:  CF-1.6
│   │   │           title:        ISMIP6 Projections Greenland model output
│   │   │           institution:  Alfred Wegener Institute for Polar and Marine Research, DE,...
│   │   │           source:       PISM
│   │   │           references:   https://doi.org/10.5194/tc-14-3033-2020
│   │   │           contact:      Name = Thomas Kleiner, Johannes Sutter, Angelika Humbert, E...
│   │   │           comment:      AWI, PISM1
│   │   └── Group: /AWI_PISM1/ctrl_proj_std/sftgrf
│   │           Dimensions:    (time: 86, bnds: 2, y: 761, x: 761)
│   │           Coordinates:
│   │             * time       (time) object 688B 2016-01-01 00:00:00 ... 2101-01-01 00:00:00
│   │             * bnds       (bnds) float32 8B 0.0 0.0
│   │             * y          (y) float64 6kB -3.04e+06 -3.032e+06 ... 3.032e+06 3.04e+06
│   │             * x          (x) float64 6kB -3.04e+06 -3.032e+06 ... 3.032e+06 3.04e+06
│   │           Data variables:
│   │               time_bnds  (time, bnds) float32 688B ...
│   │               sftgrf     (time, y, x) float32 199MB ...
│   │           Attributes:
│   │               Conventions:  CF-1.6
│   │               title:        ISMIP6 Projections Greenland model output
│   │               institution:  Alfred Wegener Institute for Polar and Marine Research, DE,...
│   │               source:       PISM
│   │               references:   https://doi.org/10.5194/tc-14-3033-2020
│   │               contact:      Name = Thomas Kleiner, Johannes Sutter, Angelika Humbert, E...
│   │               comment:      AWI, PISM1
│   └── Group: /AWI_PISM1/exp05
│       ├── Group: /AWI_PISM1/exp05/base
│       │       Dimensions:    (time: 86, y: 761, x: 761, bnds: 2)
│       │       Coordinates:
│       │         * time       (time) object 688B 2016-01-01 00:00:00 ... 2101-01-01 00:00:00
│       │         * y          (y) float64 6kB -3.04e+06 -3.032e+06 ... 3.032e+06 3.04e+06
│       │         * x          (x) float64 6kB -3.04e+06 -3.032e+06 ... 3.032e+06 3.04e+06
│       │         * bnds       (bnds) float32 8B 0.0 0.0
│       │       